In [1]:
pip install sentence-transformers scipy numpy

Code running zone

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer
from scipy.linalg import orthogonal_procrustes
from scipy.stats import mannwhitneyu, ttest_ind

model = SentenceTransformer('all-MiniLM-L6-v2')

# ============================================================
# Paired concepts: shared functional basis (n = 15)
# ============================================================

paired_philosopher = [
    "a valid argument preserves truth from its premises to its conclusion",
    "in some possible world the proposition fails to hold",
    "the expression refers to its semantic value relative to a model",
    "the meaning of the whole is determined by its parts and their mode of combination",
    "the ship of Theseus retains its identity through the replacement of its parts",
    "a counterfactual evaluates what would obtain under a contrary supposition",
    "two expressions are intersubstitutable salva veritate when they share extension",
    "an inference is justified when it follows from accepted premises by valid rules",
    "the concept is grounded when its application conditions are publicly accessible",
    "knowledge requires a justified belief that tracks the truth across nearby worlds",
    "the universal generalization holds when every instance satisfies the predicate",
    "a definition is non-circular when the definiens does not presuppose the definiendum",
    "a deductive proof transmits warrant from the premises to every theorem derived from them",
    "a category exhibits closure when every operation on its members yields another member",
    "consistency requires that no proposition and its negation both follow from the axioms",
]

paired_engineering = [
    "a sound structural design transfers loads from the applied force to the support",
    "the failure analysis identifies a scenario in which the component does not hold",
    "the symbol on the schematic refers to a specific physical component in the assembly",
    "the system behavior is determined by its components and how they are connected",
    "the bridge retains its function through the replacement of individual structural members",
    "a what-if analysis evaluates how the system would behave under altered conditions",
    "two components are interchangeable when they meet the same specification",
    "a design decision is justified when it follows from requirements through engineering reasoning",
    "the requirement is well-defined when its acceptance criteria are publicly verifiable",
    "a robust design maintains performance across nearby variations in operating conditions",
    "the safety requirement holds when every operating condition satisfies the limit",
    "a specification is non-circular when its definition does not presuppose its own satisfaction",
    "a verification chain transmits assurance from input checks to every downstream subsystem",
    "a subsystem exhibits closure when every internal operation produces an admissible state",
    "consistency requires that no requirement and its negation are both imposed on the design",
]

# ============================================================
# Jargon: concepts living sharply in one basis (n = 5)
# ============================================================

philosopher_only = [
    "the qualia of phenomenal red are intrinsically private and ineffable",
    "transcendental idealism conditions the very form of sensible intuition",
    "the de re reading attributes the property to the object itself rather than to the mode of presentation",
    "supervenience without reduction allows mental properties to depend on physical ones without identity",
    "Husserlian epoché brackets the natural attitude in order to attend to phenomena as given",
]

engineering_only = [
    "the von Mises stress exceeds the yield strength at the notch root",
    "the Reynolds number governs the transition from laminar to turbulent flow",
    "fatigue crack growth follows the Paris law under cyclic loading",
    "the bode plot shows the frequency response of the closed-loop system",
    "the heat affected zone around the weld develops a coarse grain microstructure",
]

# ============================================================
# Embed
# ============================================================

V_phil      = model.encode(paired_philosopher)
V_eng       = model.encode(paired_engineering)
V_phil_only = model.encode(philosopher_only)
V_eng_only  = model.encode(engineering_only)

# ============================================================
# Estimate the interpretation map T: phil -> eng
# ============================================================

M, _ = orthogonal_procrustes(V_phil, V_eng)

# ============================================================
# Helpers
# ============================================================

def subspace_decomposition(vectors, basis_vectors):
    U, S, Vt = np.linalg.svd(basis_vectors, full_matrices=False)
    rank = int(np.sum(S > 1e-6))
    Q = Vt[:rank]
    projected = vectors @ Q.T @ Q
    residual  = vectors - projected
    return (np.linalg.norm(projected, axis=1) ** 2,
            np.linalg.norm(residual,  axis=1) ** 2,
            np.linalg.norm(vectors,   axis=1) ** 2)

n_paired = len(paired_philosopher)

# --- Paired LOO, forward (phil -> eng) ---
loo_within_fwd  = np.zeros(n_paired)
loo_outside_fwd = np.zeros(n_paired)
loo_total_fwd   = np.zeros(n_paired)
loo_recon_fwd   = np.zeros(n_paired)

for i in range(n_paired):
    mask = np.arange(n_paired) != i
    mapped_i = V_phil[i:i+1] @ M
    w, o, t = subspace_decomposition(mapped_i, V_eng[mask])
    loo_within_fwd[i], loo_outside_fwd[i], loo_total_fwd[i] = w[0], o[0], t[0]
    loo_recon_fwd[i] = np.linalg.norm(V_eng[i] - mapped_i[0])

# --- Paired LOO, reverse (eng -> phil) ---
loo_within_rev  = np.zeros(n_paired)
loo_outside_rev = np.zeros(n_paired)
loo_total_rev   = np.zeros(n_paired)
loo_recon_rev   = np.zeros(n_paired)

for i in range(n_paired):
    mask = np.arange(n_paired) != i
    mapped_i = V_eng[i:i+1] @ M.T
    w, o, t = subspace_decomposition(mapped_i, V_phil[mask])
    loo_within_rev[i], loo_outside_rev[i], loo_total_rev[i] = w[0], o[0], t[0]
    loo_recon_rev[i] = np.linalg.norm(V_phil[i] - mapped_i[0])

# --- Jargon ---
mapped_phil_only = V_phil_only @ M
po_within, po_outside, po_total = subspace_decomposition(mapped_phil_only, V_eng)

mapped_eng_only = V_eng_only @ M.T
eo_within, eo_outside, eo_total = subspace_decomposition(mapped_eng_only, V_phil)

# ============================================================
# Format and report
# ============================================================

def fmt_pair(within, outside, total):
    w_frac = within / total
    o_frac = outside / total
    return (f"{w_frac.mean():.3f} ± {w_frac.std():.3f}",
            f"{o_frac.mean():.3f} ± {o_frac.std():.3f}",
            f"{o_frac.mean()*100:.1f}%")

paired_w_fwd, paired_o_fwd, paired_pct_fwd = fmt_pair(loo_within_fwd, loo_outside_fwd, loo_total_fwd)
paired_w_rev, paired_o_rev, paired_pct_rev = fmt_pair(loo_within_rev, loo_outside_rev, loo_total_rev)
po_w, po_o, po_pct = fmt_pair(po_within, po_outside, po_total)
eo_w, eo_o, eo_pct = fmt_pair(eo_within, eo_outside, eo_total)

print(f"\n{'Concept type':<42} {'Recon err':>17} {'Within (frac)':>17} {'Outside (frac)':>17} {'Outside %':>10}")
print("-" * 107)
print(f"{'Paired, LOO (φ → Eng)':<42} "
      f"{f'{loo_recon_fwd.mean():.3f} ± {loo_recon_fwd.std():.3f}':>17} "
      f"{paired_w_fwd:>17} {paired_o_fwd:>17} {paired_pct_fwd:>10}")
print(f"{'Paired, LOO (Eng → φ)':<42} "
      f"{f'{loo_recon_rev.mean():.3f} ± {loo_recon_rev.std():.3f}':>17} "
      f"{paired_w_rev:>17} {paired_o_rev:>17} {paired_pct_rev:>10}")
print(f"{'Philosopher-only jargon (φ → Eng)':<42} {'—':>17} "
      f"{po_w:>17} {po_o:>17} {po_pct:>10}")
print(f"{'Engineering-only jargon (Eng → φ)':<42} {'—':>17} "
      f"{eo_w:>17} {eo_o:>17} {eo_pct:>10}")

print(f"\nN paired = {n_paired}, "
      f"N phil-only = {len(philosopher_only)}, "
      f"N eng-only = {len(engineering_only)}, "
      f"embedding dim = {V_phil.shape[1]}")
print("Note: fractions are of squared L^2 mass; within + outside ≈ 1 by orthogonality.")

# ============================================================
# Statistical tests
# ============================================================

paired_frac_fwd = loo_outside_fwd / loo_total_fwd
paired_frac_rev = loo_outside_rev / loo_total_rev
phil_only_frac  = po_outside / po_total
eng_only_frac   = eo_outside / eo_total

print("\n--- Statistical tests (one-sided: jargon outside-fraction > paired outside-fraction) ---\n")

# φ → Eng direction
u_fwd, p_mw_fwd = mannwhitneyu(phil_only_frac, paired_frac_fwd, alternative='greater')
t_fwd, p_t_fwd  = ttest_ind(phil_only_frac, paired_frac_fwd, equal_var=False, alternative='greater')
print(f"φ → Eng: jargon ({phil_only_frac.mean():.3f}) vs paired LOO ({paired_frac_fwd.mean():.3f})")
print(f"   Mann-Whitney U: U = {u_fwd:.1f}, one-sided p = {p_mw_fwd:.4f}")
print(f"   Welch's t-test: t = {t_fwd:.3f}, one-sided p = {p_t_fwd:.4f}")

# Eng → φ direction
u_rev, p_mw_rev = mannwhitneyu(eng_only_frac, paired_frac_rev, alternative='greater')
t_rev, p_t_rev  = ttest_ind(eng_only_frac, paired_frac_rev, equal_var=False, alternative='greater')
print(f"\nEng → φ: jargon ({eng_only_frac.mean():.3f}) vs paired LOO ({paired_frac_rev.mean():.3f})")
print(f"   Mann-Whitney U: U = {u_rev:.1f}, one-sided p = {p_mw_rev:.4f}")
print(f"   Welch's t-test: t = {t_rev:.3f}, one-sided p = {p_t_rev:.4f}")

# Pooled across both directions
paired_pooled = np.concatenate([paired_frac_fwd, paired_frac_rev])
jargon_pooled = np.concatenate([phil_only_frac, eng_only_frac])
u_p, p_mw_p = mannwhitneyu(jargon_pooled, paired_pooled, alternative='greater')
t_p, p_t_p  = ttest_ind(jargon_pooled, paired_pooled, equal_var=False, alternative='greater')
print(f"\nPooled (both directions): jargon ({jargon_pooled.mean():.3f}) vs paired ({paired_pooled.mean():.3f})")
print(f"   Mann-Whitney U: U = {u_p:.1f}, one-sided p = {p_mw_p:.4f}")
print(f"   Welch's t-test: t = {t_p:.3f}, one-sided p = {p_t_p:.4f}")

# ============================================================
# Per-concept diagnostic
# ============================================================

print("\nPer-concept LOO outside-fraction (φ → Eng):")
for i, (sent, out_sq, tot_sq) in enumerate(zip(paired_philosopher, loo_outside_fwd, loo_total_fwd)):
    frac = out_sq / tot_sq
    label = sent[:64] + ("..." if len(sent) > 64 else "")
    print(f"  [{i:2d}] {frac*100:5.1f}%   {label}")

print("\nPer-concept LOO outside-fraction (Eng → φ):")
for i, (sent, out_sq, tot_sq) in enumerate(zip(paired_engineering, loo_outside_rev, loo_total_rev)):
    frac = out_sq / tot_sq
    label = sent[:64] + ("..." if len(sent) > 64 else "")
    print(f"  [{i:2d}] {frac*100:5.1f}%   {label}")